In [ ]:
import joblib

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from gensim.models import Word2Vec

In [120]:
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Предобработка данных

In [121]:
df = pd.read_csv(r'data\Petitions.csv').drop(['id'], axis=1)

df_small = df.sample(n=3000, random_state=RANDOM_SEED)
df_small.to_csv(r'data\small_petitions.csv', index=False)

In [122]:
df = pd.read_csv(r'data\small_petitions.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 2 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   public_petition_text  3000 non-null   object
 1   reason_category       3000 non-null   object
dtypes: object(2)
memory usage: 47.0+ KB


In [123]:
df['reason_category'].value_counts()

reason_category
Благоустройство                                                                     1764
Содержание МКД                                                                       714
Нарушение правил пользования общим имуществом                                         99
Незаконная информационная и (или) рекламная конструкция                               86
Фасад                                                                                 67
Повреждения или неисправность элементов уличной инфраструктуры                        58
Кровля                                                                                47
Состояние рекламных или информационных конструкций                                    41
Водоснабжение                                                                         41
Незаконная реализация товаров с торгового оборудования (прилавок, ящик, с земли)      18
Санитарное состояние                                                                  18
Центр

In [124]:
from nlp_module.text_preprocess import full_text_preprocessing

df['processed_tokens'] = df['public_petition_text'].apply(full_text_preprocessing)
df.head()

,public_petition_text,reason_category,processed_tokens
0,На газоне разбросан различный мусор. \nПросьба...,Благоустройство,"[газон, разбросать, различный, мусор, просьба,..."
1,"Конструкции, препятствующие парковке",Благоустройство,"[конструкция, препятствовать, парковка]"
2,Лестницу убирают раз в три месяца и просто выл...,Содержание МКД,"[лестница, убирать, месяц, просто, выливать, в..."
3,мусор с задней стороны дома,Благоустройство,"[мусор, задний, сторона, дом]"
4,Снова не работает уличный фонарь у входной две...,Содержание МКД,"[снова, работать, уличный, фонарь, входной, дв..."


In [125]:
texts = df['processed_tokens'].tolist()
texts[:3]

[['газон',
  'разбросать',
  'различный',
  'мусор',
  'просьба',
  'вывезти',
  'утилизация',
  'координата'],
 ['конструкция', 'препятствовать', 'парковка'],
 ['лестница',
  'убирать',
  'месяц',
  'просто',
  'выливать',
  'ведро',
  'вода',
  'это',
  'уборка']]

In [126]:
w2v_model = Word2Vec(sentences=texts, vector_size=100, window=5, min_count=3, workers=8, sg=1, epochs=50, seed=RANDOM_SEED)
print(f"Обучено векторов: {len(w2v_model.wv.key_to_index)}")

Обучено векторов: 1547


In [127]:
def qualitative_evaluation(model, test_words):
    for word in test_words:
        if word in model.wv:
            similar = model.wv.most_similar(word, topn=3)
            print(f"Слова, связанные с '{word}':")
            for similar_word, score in similar:
                print(f"  {similar_word}: {score:.3f}")
            print()
        else:
            print(f"Слово '{word}' не найдено в словаре\n")

test_words = ['ремонт', 'мусор', 'дом', 'уборка', 'вода', 'крыша']
qualitative_evaluation(w2v_model, test_words)

Слова, связанные с 'ремонт':
  косметический: 0.619
  капитальный: 0.619
  перечень: 0.563

Слова, связанные с 'мусор':
  пластик: 0.631
  мелкий: 0.615
  окурок: 0.609

Слова, связанные с 'дом':
  загородный: 0.513
  парашютный: 0.474
  корпус: 0.469

Слова, связанные с 'уборка':
  влажный: 0.663
  подметание: 0.661
  подметать: 0.654

Слова, связанные с 'вода':
  холодный: 0.614
  горячий: 0.613
  напор: 0.600

Слова, связанные с 'крыша':
  протекать: 0.630
  несмотря: 0.609
  неоднократный: 0.587



In [128]:
w2v_model.save('w2v_sg_small.bin')

In [129]:
df['token_vectors'] = df['processed_tokens'].apply(lambda tokens: [w2v_model.wv[token] if token in w2v_model.wv else np.zeros(100) for token in tokens])

In [130]:
df.head()

,public_petition_text,reason_category,processed_tokens,token_vectors
0,На газоне разбросан различный мусор. \nПросьба...,Благоустройство,"[газон, разбросать, различный, мусор, просьба,...","[[-0.33424243, -0.11051619, -0.19500859, -0.08..."
1,"Конструкции, препятствующие парковке",Благоустройство,"[конструкция, препятствовать, парковка]","[[-0.1846369, -0.54203767, -0.28949848, 0.4900..."
2,Лестницу убирают раз в три месяца и просто выл...,Содержание МКД,"[лестница, убирать, месяц, просто, выливать, в...","[[0.14613041, -0.15402068, -0.6182528, -0.3854..."
3,мусор с задней стороны дома,Благоустройство,"[мусор, задний, сторона, дом]","[[0.15385777, -0.17964008, 0.5023354, -0.30284..."
4,Снова не работает уличный фонарь у входной две...,Содержание МКД,"[снова, работать, уличный, фонарь, входной, дв...","[[0.6580241, -0.054691114, -0.1804161, 0.04438..."


In [131]:
le = LabelEncoder()
df['category_encoded'] = le.fit_transform(df['reason_category'])

joblib.dump(le, 'label_encoder.pkl')
df.head()

,public_petition_text,reason_category,processed_tokens,token_vectors,category_encoded
0,На газоне разбросан различный мусор. \nПросьба...,Благоустройство,"[газон, разбросать, различный, мусор, просьба,...","[[-0.33424243, -0.11051619, -0.19500859, -0.08...",0
1,"Конструкции, препятствующие парковке",Благоустройство,"[конструкция, препятствовать, парковка]","[[-0.1846369, -0.54203767, -0.28949848, 0.4900...",0
2,Лестницу убирают раз в три месяца и просто выл...,Содержание МКД,"[лестница, убирать, месяц, просто, выливать, в...","[[0.14613041, -0.15402068, -0.6182528, -0.3854...",11
3,мусор с задней стороны дома,Благоустройство,"[мусор, задний, сторона, дом]","[[0.15385777, -0.17964008, 0.5023354, -0.30284...",0
4,Снова не работает уличный фонарь у входной две...,Содержание МКД,"[снова, работать, уличный, фонарь, входной, дв...","[[0.6580241, -0.054691114, -0.1804161, 0.04438...",11


In [132]:
for i, category in enumerate(le.classes_):
    print(f"{category} : {i}")

Благоустройство : 0
Водоотведение : 1
Водоснабжение : 2
Кровля : 3
Нарушение порядка пользования общим имуществом : 4
Нарушение правил пользования общим имуществом : 5
Незаконная информационная и (или) рекламная конструкция : 6
Незаконная реализация товаров с торгового оборудования (прилавок, ящик, с земли) : 7
Повреждения или неисправность элементов уличной инфраструктуры : 8
Подвалы : 9
Санитарное состояние : 10
Содержание МКД : 11
Состояние рекламных или информационных конструкций : 12
Фасад : 13
Центральное отопление : 14


In [133]:
df.to_csv('data/all_data.csv')

In [134]:
df = df[['token_vectors', 'category_encoded']].copy()
df.head()

,token_vectors,category_encoded
0,"[[-0.33424243, -0.11051619, -0.19500859, -0.08...",0
1,"[[-0.1846369, -0.54203767, -0.28949848, 0.4900...",0
2,"[[0.14613041, -0.15402068, -0.6182528, -0.3854...",11
3,"[[0.15385777, -0.17964008, 0.5023354, -0.30284...",0
4,"[[0.6580241, -0.054691114, -0.1804161, 0.04438...",11


In [135]:
X = df['token_vectors']
y = df['category_encoded']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

len(X_train), len(X_val), len(X_test)

(2100, 450, 450)

# Создание моделей

In [136]:
device = torch.device('cuda')

In [137]:
def prepare_data(X, y, max_len=50, vector_size=100):
    processed_sequences = np.zeros((len(X), max_len, vector_size), dtype=np.float32)
    
    for i, sequence in enumerate(X):
        current_len = min(len(sequence), max_len)

        if len(sequence) > 0:
            sequence_array = np.array(sequence[:current_len], dtype=np.float32)
            processed_sequences[i, :current_len] = sequence_array

    X_tensor = torch.from_numpy(processed_sequences)
    y_tensor = torch.tensor(y, dtype=torch.long)
    
    return X_tensor, y_tensor


X_tensor, y_tensor = prepare_data(X_train, y_train.values)
X_tensor = X_tensor.to(device)
y_tensor = y_tensor.to(device)

dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

X_tensor_val, y_tensor_val = prepare_data(X_val, y_val.values)
X_tensor_val = X_tensor_val.to(device)
y_tensor_val = y_tensor_val.to(device)

dataset_val = TensorDataset(X_tensor_val, y_tensor_val)
dataloader_val = DataLoader(dataset_val, batch_size=32, shuffle=False)

In [138]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size=100, hidden_size=128, num_classes=15):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        _, hidden = self.rnn(x)
        return self.fc(hidden.squeeze(0))

class SimpleLSTM(nn.Module):
    def __init__(self, input_size=100, hidden_size=128, num_classes=15):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        return self.fc(hidden.squeeze(0))

class SimpleGRU(nn.Module):
    def __init__(self, input_size=100, hidden_size=128, num_classes=15):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, hidden = self.gru(x)
        return self.fc(hidden.squeeze(0))


rnn_model = SimpleRNN()
lstm_model = SimpleLSTM()
gru_model = SimpleGRU()

In [139]:
class CustomGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size

        self.W_r = nn.Linear(input_size, hidden_size)
        self.U_r = nn.Linear(hidden_size, hidden_size)

        self.W_z = nn.Linear(input_size, hidden_size)
        self.U_z = nn.Linear(hidden_size, hidden_size)

        self.W_h = nn.Linear(input_size, hidden_size)
        self.U_h = nn.Linear(hidden_size, hidden_size)

    def forward(self, x, h_prev=None):
        batch_size, seq_len, _ = x.size()
        
        if h_prev is None:
            h_prev = torch.zeros(batch_size, self.hidden_size).to(x.device)
        
        outputs = []
        h = h_prev
        
        for t in range(seq_len):
            x_t = x[:, t, :]

            r = torch.sigmoid(self.W_r(x_t) + self.U_r(h))
            z = torch.sigmoid(self.W_z(x_t) + self.U_z(h))
            h_tilde = torch.tanh(self.W_h(x_t) + self.U_h(r * h))
            h = (1 - z) * h + z * h_tilde
            outputs.append(h.unsqueeze(1))
        
        return torch.cat(outputs, dim=1), h

class CustomGRUModel(nn.Module):
    def __init__(self, input_size=100, hidden_size=128, num_classes=15):
        super().__init__()
        self.gru = CustomGRU(input_size, hidden_size)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        _, hidden = self.gru(x)
        return self.fc(hidden)


custom_gru_model = CustomGRUModel().to(device)

In [140]:
def predict_classes_from_X(model, X_data):
    X_tensor, _ = prepare_data(X_data, [])
    X_tensor = X_tensor.to(device)
    
    with torch.no_grad():
        outputs = model(X_tensor)
        _, preds = torch.max(outputs, 1)
    
    return preds.cpu().numpy()

def calculate_f1(model, dataloader):
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch_X, batch_y in dataloader:
            outputs = model(batch_X)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(batch_y.cpu().numpy())
    
    return f1_score(all_targets, all_preds, average='weighted')

def train_model(model, train_loader, val_loader, epochs=10, model_name='Model'):
    model.to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters())
    best_f1 = 0
    
    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

        current_f1 = calculate_f1(model, val_loader)

        if current_f1 > best_f1:
            best_f1 = current_f1
            torch.save(model.state_dict(), f'best_{model_name}.pth')
            print(f'!-- New best model --! F1: {current_f1:.4f}')
        
        print(f'Epoch {epoch+1}, F1: {current_f1:.4f}')

    model.load_state_dict(torch.load(f'best_{model_name}.pth', weights_only=False))
    
    return model

In [141]:
train_model(rnn_model, dataloader, dataloader_val, epochs=70, model_name="RNN")

rnn_f1 = calculate_f1(rnn_model, dataloader_val)

print(f"RNN Accuracy: {rnn_f1:.3f}")

!-- New best model --! F1: 0.4205
Epoch 1, F1: 0.4205
Epoch 2, F1: 0.4205
Epoch 3, F1: 0.4205
!-- New best model --! F1: 0.4242
Epoch 4, F1: 0.4242
!-- New best model --! F1: 0.4315
Epoch 5, F1: 0.4315
!-- New best model --! F1: 0.4357
Epoch 6, F1: 0.4357
Epoch 7, F1: 0.4309
Epoch 8, F1: 0.4315
Epoch 9, F1: 0.4357
!-- New best model --! F1: 0.4363
Epoch 10, F1: 0.4363
Epoch 11, F1: 0.4363
Epoch 12, F1: 0.4363
Epoch 13, F1: 0.4363
!-- New best model --! F1: 0.4410
Epoch 14, F1: 0.4410
Epoch 15, F1: 0.4405
Epoch 16, F1: 0.4410
Epoch 17, F1: 0.4410
Epoch 18, F1: 0.4363
Epoch 19, F1: 0.4363
Epoch 20, F1: 0.4327
Epoch 21, F1: 0.4357
Epoch 22, F1: 0.4340
Epoch 23, F1: 0.4310
!-- New best model --! F1: 0.6398
Epoch 24, F1: 0.6398
!-- New best model --! F1: 0.6626
Epoch 25, F1: 0.6626
!-- New best model --! F1: 0.6788
Epoch 26, F1: 0.6788
Epoch 27, F1: 0.6343
Epoch 28, F1: 0.5605
!-- New best model --! F1: 0.6806
Epoch 29, F1: 0.6806
!-- New best model --! F1: 0.7048
Epoch 30, F1: 0.7048
Epoch

In [142]:
train_model(lstm_model, dataloader, dataloader_val, epochs=70, model_name="LSTM") 

lstm_f1 = calculate_f1(lstm_model, dataloader_val)

print(f"LSTM Accuracy: {lstm_f1:.3f}")

!-- New best model --! F1: 0.4205
Epoch 1, F1: 0.4205
Epoch 2, F1: 0.4205
Epoch 3, F1: 0.4205
!-- New best model --! F1: 0.4362
Epoch 4, F1: 0.4362
!-- New best model --! F1: 0.4383
Epoch 5, F1: 0.4383
!-- New best model --! F1: 0.4464
Epoch 6, F1: 0.4464
!-- New best model --! F1: 0.6760
Epoch 7, F1: 0.6760
!-- New best model --! F1: 0.6827
Epoch 8, F1: 0.6827
Epoch 9, F1: 0.5074
Epoch 10, F1: 0.6074
Epoch 11, F1: 0.6410
Epoch 12, F1: 0.6461
Epoch 13, F1: 0.6302
Epoch 14, F1: 0.6509
Epoch 15, F1: 0.6588
Epoch 16, F1: 0.6016
Epoch 17, F1: 0.6091
Epoch 18, F1: 0.6458
!-- New best model --! F1: 0.7013
Epoch 19, F1: 0.7013
Epoch 20, F1: 0.6971
Epoch 21, F1: 0.4315
Epoch 22, F1: 0.6973
Epoch 23, F1: 0.6967
!-- New best model --! F1: 0.7146
Epoch 24, F1: 0.7146
Epoch 25, F1: 0.6998
Epoch 26, F1: 0.7144
Epoch 27, F1: 0.7058
Epoch 28, F1: 0.6846
Epoch 29, F1: 0.7096
Epoch 30, F1: 0.7113
Epoch 31, F1: 0.7073
!-- New best model --! F1: 0.7218
Epoch 32, F1: 0.7218
!-- New best model --! F1: 0.73

In [143]:
train_model(gru_model, dataloader, dataloader_val, epochs=70, model_name="GRU")

gru_f1 = calculate_f1(gru_model, dataloader_val)

print(f"GRU Accuracy: {gru_f1:.3f}")

!-- New best model --! F1: 0.4205
Epoch 1, F1: 0.4205
Epoch 2, F1: 0.4205
!-- New best model --! F1: 0.6902
Epoch 3, F1: 0.6902
!-- New best model --! F1: 0.7134
Epoch 4, F1: 0.7134
!-- New best model --! F1: 0.7504
Epoch 5, F1: 0.7504
!-- New best model --! F1: 0.7597
Epoch 6, F1: 0.7597
!-- New best model --! F1: 0.7963
Epoch 7, F1: 0.7963
!-- New best model --! F1: 0.8134
Epoch 8, F1: 0.8134
!-- New best model --! F1: 0.8340
Epoch 9, F1: 0.8340
Epoch 10, F1: 0.8233
!-- New best model --! F1: 0.8351
Epoch 11, F1: 0.8351
!-- New best model --! F1: 0.8377
Epoch 12, F1: 0.8377
!-- New best model --! F1: 0.8551
Epoch 13, F1: 0.8551
Epoch 14, F1: 0.8426
Epoch 15, F1: 0.8517
Epoch 16, F1: 0.8451
Epoch 17, F1: 0.8363
!-- New best model --! F1: 0.8620
Epoch 18, F1: 0.8620
Epoch 19, F1: 0.8501
Epoch 20, F1: 0.8539
Epoch 21, F1: 0.8463
Epoch 22, F1: 0.8470
Epoch 23, F1: 0.8535
Epoch 24, F1: 0.8444
!-- New best model --! F1: 0.8627
Epoch 25, F1: 0.8627
Epoch 26, F1: 0.8504
Epoch 27, F1: 0.8515


In [144]:
train_model(custom_gru_model, dataloader, dataloader_val, epochs=70, model_name="CustomGRU")

custom_gru_f1 = calculate_f1(custom_gru_model, dataloader_val)

print(f"RNN Accuracy: {custom_gru_f1:.3f}")

!-- New best model --! F1: 0.4205
Epoch 1, F1: 0.4205
!-- New best model --! F1: 0.4331
Epoch 2, F1: 0.4331
!-- New best model --! F1: 0.7116
Epoch 3, F1: 0.7116
Epoch 4, F1: 0.7064
!-- New best model --! F1: 0.7461
Epoch 5, F1: 0.7461
!-- New best model --! F1: 0.7531
Epoch 6, F1: 0.7531
!-- New best model --! F1: 0.7775
Epoch 7, F1: 0.7775
!-- New best model --! F1: 0.8028
Epoch 8, F1: 0.8028
!-- New best model --! F1: 0.8254
Epoch 9, F1: 0.8254
Epoch 10, F1: 0.8148
!-- New best model --! F1: 0.8329
Epoch 11, F1: 0.8329
Epoch 12, F1: 0.8206
!-- New best model --! F1: 0.8344
Epoch 13, F1: 0.8344
!-- New best model --! F1: 0.8547
Epoch 14, F1: 0.8547
Epoch 15, F1: 0.8393
Epoch 16, F1: 0.8304
Epoch 17, F1: 0.8464
Epoch 18, F1: 0.8511
Epoch 19, F1: 0.8445
Epoch 20, F1: 0.8463
Epoch 21, F1: 0.8450
Epoch 22, F1: 0.8527
Epoch 23, F1: 0.8500
!-- New best model --! F1: 0.8581
Epoch 24, F1: 0.8581
Epoch 25, F1: 0.8468
Epoch 26, F1: 0.8427
Epoch 27, F1: 0.8511
Epoch 28, F1: 0.8527
!-- New best 

In [145]:
torch.cuda.empty_cache()

In [149]:
def predict_classes(model, X_tensor):
    with torch.no_grad():
        outputs = model(X_tensor)
        _, preds = torch.max(outputs, 1)
    return preds.cpu().numpy()

def evaluate_all_models(models_dict, X_tensor_test, y_test):
    results = {}
    
    for name, model in models_dict.items():
        preds = predict_classes(model, X_tensor_test)
        
        results[name] = {
            'accuracy': accuracy_score(y_test, preds),
            'f1': f1_score(y_test, preds, average='weighted', zero_division=0),
            'precision': precision_score(y_test, preds, average='weighted', zero_division=0),
            'recall': recall_score(y_test, preds, average='weighted', zero_division=0)
        }

    print(f"{'Модель':<8} {'Acc':<8} {'F1':<8} {'Prec':<8} {'Rec':<8}")
    print("-" * 41)
    for name, metrics in results.items():
        print(f"{name:<8} {metrics['accuracy']:.3f}    {metrics['f1']:.3f}    "
              f"{metrics['precision']:.3f}    {metrics['recall']:.3f}")
    
    return results

X_tensor_test, _ = prepare_data(X_test, y_test.values)
X_tensor_test = X_tensor_test.to(device)

models = {'RNN': rnn_model, 'LSTM': lstm_model, 'GRU': gru_model, 'C_GRU': custom_gru_model}
results = evaluate_all_models(models, X_tensor_test, y_test)

Модель   Acc      F1       Prec     Rec     
-----------------------------------------
RNN      0.742    0.702    0.674    0.742
LSTM     0.818    0.799    0.784    0.818
GRU      0.838    0.830    0.834    0.838
C_GRU    0.833    0.828    0.843    0.833
